## 1. Import packages

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

## 2. Load RNA summaries and Wilcoxon results

In [5]:
goblet_rna = pd.read_csv("rna_summaries/GSE229202_goblet_RNA_summary.csv")
club_rna = pd.read_csv("rna_summaries/GSE229202_club_RNA_summary.csv")
multiciliated_rna = pd.read_csv("rna_summaries/GSE229202_ciliated_RNA_summary.csv")

wilcoxon = pd.read_csv("/rna_summaries/GSE229202_wilcoxon_DE.csv")

In [6]:
print(goblet_rna.shape)
print(club_rna.shape)
print(multiciliated_rna.shape)
print(wilcoxon.shape)

(20266, 11)
(20266, 11)
(20266, 11)
(60798, 7)


## 2. Load proteomics data

In [11]:

goblet_protein = pd.read_csv("ver_3_goblet_sig_hits_results.csv")
club_protein = pd.read_csv("ver_3_secretory_sig_hits_results.csv")
multiciliated_protein = pd.read_csv("ver_3_mccs_sig_hits_results.csv")

print(goblet_protein.columns)
print(goblet_protein.head())

Index(['Protein.Group', 'Protein.Names', 'Genes', 'Description',
       'Log2.Mean.ctrl', 'Log2.Mean.il13', 'Log2.Mean', 'Log2.Difference',
       'p.value', 'Rank', 'BH.critical.value', 'Significance',
       'Significant.up..1', 'Significant.down..1', 'Significant.up..0.5',
       'Significant.down..0.5', 'observed_ctrl', 'observed_il13',
       'flag_ctrl_high_missing', 'flag_il13_high_missing',
       'flag_ctrl_sided_absent', 'flag_il13_sided_absent', 'class', 'theme',
       'abundance_bin', 'ctrl_abundance_bin', 'il13_abundance_bin',
       'mean_abundance_bin'],
      dtype='object')
  Protein.Group Protein.Names    Genes  \
0        Q8IYE1   CCD13_HUMAN   CCDC13   
1        Q5TID7   CC181_HUMAN  CCDC181   
2        Q8N5R6   CCD33_HUMAN   CCDC33   
3        A2VCK2   DCD2B_HUMAN   DCDC2B   
4        Q6P656   CF161_HUMAN  CFAP161   

                                  Description  Log2.Mean.ctrl  Log2.Mean.il13  \
0    Coiled-coil domain-containing protein 13       20.276064      

In [12]:
protein_columns = [
    "Genes",
    "Protein.Group",
    "Protein.Names",
    "Description",
    "Log2.Mean.ctrl",
    "Log2.Mean.il13",
    "Log2.Mean",
    "Log2.Difference",
    "p.value",
    "Rank",
    "Significance",
    "observed_ctrl",
    "observed_il13",
    "class"
]

def prepare_protein_data(df):

    df = df[protein_columns].copy()

    df = df.rename(columns={
        "Genes": "gene",
        "Protein.Group": "protein_group",
        "Protein.Names": "protein_names",
        "Description": "protein_description",
        "Log2.Mean.ctrl": "protein_log2_control",
        "Log2.Mean.il13": "protein_log2_il13",
        "Log2.Mean": "protein_log2_mean",
        "Log2.Difference": "protein_log2FC",
        "p.value": "protein_pvalue",
        "Rank": "protein_rank",
        "Significance": "protein_significance",
        "class": "protein_missingness_class"
    })

    return df

In [13]:
goblet_protein = prepare_protein_data(goblet_protein)
club_protein = prepare_protein_data(club_protein)
multiciliated_protein = prepare_protein_data(multiciliated_protein)

In [14]:
# before merging, make copy of proteomics dataframe
goblet_protein_full = goblet_protein.copy()
club_protein_full = club_protein.copy()
multiciliated_protein_full = multiciliated_protein.copy()

## 3. Deal with ambiuous proteins

In [15]:
goblet_protein_clean = goblet_protein[
    ~goblet_protein["gene"].str.contains(";", na=False)
].copy()

club_protein_clean = club_protein[
    ~club_protein["gene"].str.contains(";", na=False)
].copy()

multiciliated_protein_clean = multiciliated_protein[
    ~multiciliated_protein["gene"].str.contains(";", na=False)
].copy()

In [16]:
# similar to jackson et al integration, if several protein groups map to the same gene, retain the one with the highest control protein abundance
goblet_protein_clean = (
    goblet_protein_clean
    .sort_values("protein_log2_control", ascending=False)
    .drop_duplicates(subset="gene", keep="first")
    .copy()
)

club_protein_clean = (
    club_protein_clean
    .sort_values("protein_log2_control", ascending=False)
    .drop_duplicates(subset="gene", keep="first")
    .copy()
)

multiciliated_protein_clean = (
    multiciliated_protein_clean
    .sort_values("protein_log2_control", ascending=False)
    .drop_duplicates(subset="gene", keep="first")
    .copy()
)

In [17]:
def make_ambiguous_gene_table(protein_df, rna_df, celltype):
    # keep only protein groups containing ;
    ambig = protein_df[protein_df["gene"].notna() & protein_df["gene"].str.contains(";", na=False)].copy()

    # keep original protein group gene string
    ambig["protein_gene_group"] = ambig["gene"]

    # split into separate genes
    ambig["gene"] = ambig["gene"].str.split(";")
    ambig = ambig.explode("gene").copy()

    # remove spaces if any
    ambig["gene"] = ambig["gene"].str.strip()

    # merge each split gene with its RNA expression
    out = ambig.merge(
        rna_df,
        on="gene",
        how="left",
        suffixes=("", f"_{celltype}_rna")
    )

    out["cell_type_protein"] = celltype

    return out

In [21]:
goblet_ambiguous = make_ambiguous_gene_table(
    goblet_protein_full,
    goblet_rna,
    "goblet"
)

club_ambiguous = make_ambiguous_gene_table(
    club_protein_full,
    club_rna,
    "club"
)

multiciliated_ambiguous = make_ambiguous_gene_table(
    multiciliated_protein_full,
    multiciliated_rna,
    "multiciliated"
)

In [24]:
def resolve_ambiguous_protein_groups(df):
    df = df.copy()
    summaries = []
    for group_name, g in df.groupby("protein_gene_group"):
        # the protein gene group
        g = g.copy()
        # count candidate genes e.g. P60709;P63261 is 2
        n_candidates = len(g)
        # count how many have RNA mean
        n_with_rna = g["rna_mean_ctrl"].notna().sum()
        n_missing_rna = g["rna_mean_ctrl"].isna().sum()
        # default outputs
        assigned_genes = [] # list of genes that will receive the protein value
        resolved_gene = np.nan # summary of the assigned gene or genes
        resolution_rule = "unresolved" # why the decision was mad e.g. 5x_dominant, partial_rna_single_candidate, shared_similar_rna
        ambiguous_flag = True

        # genes with RNA data only
        g_rna = g[g["rna_mean_ctrl"].notna()].copy()

        # ------------------------
        # Case 0: no RNA
        # ------------------------
        if n_with_rna == 0:
            assigned_genes = []
            resolved_gene = np.nan
            resolution_rule = "no_rna_data"

        # ------------------------
        # Case 1: only one RNA gene
        # ------------------------
        elif n_with_rna == 1: # e.g. GATD3 = 0.7, GATD3B = NaN -> so n_with_rna == 1 is TRUE
            assigned_genes = g_rna["gene"].tolist()
            resolved_gene = assigned_genes[0] # this stores it as a single value e.g. GATD3
            resolution_rule = "partial_rna_single_candidate" # only one candidate gene had RNA data, so protein is tentatively assigned to that gene

        # ------------------------
        # Case 2: multiple RNA genes - all or at least 2 candidates have RNA data
        # ------------------------
        else:
            # sort by increasing RNA mean e.g. H3-3A = 2.157, H3C1 = 0.001, H3C15 = 0.000
            g_sorted = g_rna.sort_values("rna_mean_ctrl", ascending=False)
            # get the top gene and the rna_mean (first row) e.g. top_gene = "H3-3A", top_rna = 2.157
            top_gene = g_sorted["gene"].iloc[0]
            top_rna = g_sorted["rna_mean_ctrl"].iloc[0]
            # get the next top gene and the rna_mean
            # if there is more than one candidate, this gets the second-highest RNA e.g. second_rna = 0.0001
            second_rna = g_sorted["rna_mean_ctrl"].iloc[1]
            
            # 1. all candidates have RNA mean value (complete)
            # Rule: top gene must be at least 5x higher and > 0.1 than next highest - gets rid of bias due to small number
            if n_with_rna == n_candidates:
                if (top_rna >= 5 * (second_rna + 1e-9)) and (top_rna >= 0.1): # adding 1e-9 avoids errors/issues where second_rna is 0.000, so I just add a tiny value here
                    # e.g. here 2.157 (top_rna) bigger than 0.005 (second_rna calulcation value = 0.001), and bigger than 0.1, so H3-3A is dominant
                    assigned_genes = [top_gene]
                    resolved_gene = top_gene
                    resolution_rule = f"complete_5x_dominant"
                else: # Otherwise RNA is too similar; assign protein value to all genes (when all candidate genes have RNA data, but no gene is 5× higher than the next)
                    assigned_genes = g["gene"].tolist() # e.g. ["MAGOH", "MAGOHB"]
                    resolved_gene = ";".join(assigned_genes) # combines to one string e.g. MAGOH;MAGOHB
                    resolution_rule = "complete_shared_similar_rna" #  carry the protein value forward for all candidates

            # 2. for groups where not all candidates have RNA data (partial) e.g. A = 0.7, B = 0.1, C = NaN
            else:
                if (top_rna >= 5 * (second_rna + 1e-9)) and (top_rna >= 0.1): # adding 1e-9 avoids errors/issues where second_rna is 0.000, so I just add a tiny value here
                    # e.g. here 2.157 (top_rna) ≥ 0.005 (second_rna calulcation value), so H3-3A is dominant
                    assigned_genes = [top_gene]
                    resolved_gene = top_gene
                    resolution_rule = f"partial_5x_dominant"
                else: # Otherwise RNA is too similar; assign protein value to all genes (when all candidate genes have RNA data, but no gene is 5× higher than the next)
                    assigned_genes = g_rna["gene"].tolist() # e.g. ["MAGOH", "MAGOHB"]
                    resolved_gene = ";".join(assigned_genes) # combines to one string e.g. MAGOH;MAGOHB
                    resolution_rule = "partial_shared_similar_rna" #  carry the protein value forward for all candidates

        summaries.append({
            "protein_gene_group": group_name,
            "n_candidates": n_candidates,
            "n_with_rna": n_with_rna,
            "n_missing_rna": n_missing_rna,
            "resolved_gene": resolved_gene,
            "assigned_genes": ";".join(assigned_genes),
            "resolution_rule": resolution_rule,
            "ambiguous_flag": ambiguous_flag
        })

    summary_df = pd.DataFrame(summaries)
    # -----------------------------------
    # Expanded table for merging
    # -----------------------------------
    expanded_df = summary_df.copy()
    expanded_df["assigned_gene"] = (expanded_df["assigned_genes"].fillna("").str.split(";"))
    expanded_df = expanded_df.explode("assigned_gene")
    expanded_df = expanded_df[expanded_df["assigned_gene"] != ""].copy()
    expanded_df = expanded_df.rename(columns={ "protein_gene_group": "original_protein_gene_group"})
    expanded_df = expanded_df[["original_protein_gene_group","assigned_gene","resolution_rule","ambiguous_flag"]].copy()

    return expanded_df

In [25]:
goblet_ambiguous_resolved = resolve_ambiguous_protein_groups(goblet_ambiguous)

club_ambiguous_resolved = resolve_ambiguous_protein_groups(club_ambiguous)

multiciliated_ambiguous_resolved = resolve_ambiguous_protein_groups(multiciliated_ambiguous)

In [26]:
# quick check
print("Goblet:", len(goblet_ambiguous_resolved))
print("Club:", len(club_ambiguous_resolved))
print("Multiciliated:", len(multiciliated_ambiguous_resolved))

Goblet: 79
Club: 79
Multiciliated: 78


### Combine unambiguous and resolved protein-gene assignments

In [28]:
def add_ambiguous_assignments_to_clean_protein(clean_protein_df, full_protein_df, ambiguous_expanded_df):
    clean = clean_protein_df.copy()
    full = full_protein_df.copy()
    expanded = ambiguous_expanded_df.copy()

    # mark clean/unambiguous proteins
    clean["from_ambiguous_protein_group"] = False
    clean["original_protein_gene_group"] = clean["gene"]
    clean["assigned_gene"] = clean["gene"]
    clean["resolution_rule"] = "unambiguous"
    clean["ambiguous_flag"] = False

    # get original ambiguous protein abundance rows
    ambiguous_protein_values = full[full["gene"].str.contains(";", na=False)].copy()
    ambiguous_protein_values = ambiguous_protein_values.rename(columns={"gene": "original_protein_gene_group"})

    # attach protein values to assigned genes
    ambig_assigned = expanded.merge(ambiguous_protein_values,on="original_protein_gene_group",how="left")

    # now assigned_gene becomes the actual gene for downstream merging
    ambig_assigned["gene"] = ambig_assigned["assigned_gene"]
    ambig_assigned["from_ambiguous_protein_group"] = True

    # match clean columns
    keep_cols = clean.columns
    ambig_assigned = ambig_assigned[keep_cols]

    # combine ckean(unambiguous) + resolved/assigned ambiguous tables
    combined = pd.concat([clean, ambig_assigned],ignore_index=True)

    # if same gene appears multiple times, keep highest control abundance
    combined = (
        combined
        .sort_values("protein_log2_control", ascending=False)
        .drop_duplicates(subset="gene", keep="first")
        .copy()
    )

    return combined

In [29]:
goblet_protein_final = add_ambiguous_assignments_to_clean_protein(
    goblet_protein_clean,
    goblet_protein_full,
    goblet_ambiguous_resolved
)

club_protein_final = add_ambiguous_assignments_to_clean_protein(
    club_protein_clean,
    club_protein_full,
    club_ambiguous_resolved
)

multiciliated_protein_final = add_ambiguous_assignments_to_clean_protein(
    multiciliated_protein_clean,
    multiciliated_protein_full,
    multiciliated_ambiguous_resolved
)

## 4. Merge proteomics with GSE229202 RNA expression

In [30]:
goblet_merged = goblet_protein_final.merge(
    goblet_rna,
    on="gene",
    how="inner"
)

club_merged = club_protein_final.merge(
    club_rna,
    on="gene",
    how="inner"
)

multiciliated_merged = multiciliated_protein_final.merge(
    multiciliated_rna,
    on="gene",
    how="inner"
)

In [31]:
print("Goblet matched genes:", len(goblet_merged))
print("Club matched genes:", len(club_merged))
print("Multiciliated matched genes:", len(multiciliated_merged))

Goblet matched genes: 9692
Club matched genes: 9692
Multiciliated matched genes: 9691


In [32]:
goblet_de = wilcoxon[
    wilcoxon["cell_type"] == "goblet"
].copy()

club_de = wilcoxon[
    wilcoxon["cell_type"] == "club"
].copy()

multiciliated_de = wilcoxon[
    wilcoxon["cell_type"] == "ciliated"
].copy()

In [33]:
de_columns = [
    "gene",
    "rna_log2FC",
    "rna_pvalue",
    "rna_padj",
    "rna_sig"
]

goblet_merged = goblet_merged.merge(
    goblet_de[de_columns],
    on="gene",
    how="left"
)

club_merged = club_merged.merge(
    club_de[de_columns],
    on="gene",
    how="left"
)

multiciliated_merged = multiciliated_merged.merge(
    multiciliated_de[de_columns],
    on="gene",
    how="left"
)

## 5. Assign RNA and protein directions

In [34]:
def assign_directions(df):

    df = df.copy()
    # -----------------------------
    # RNA direction
    # -----------------------------
    df["rna_direction"] = "No change"

    df.loc[
        (df["rna_sig"] == True)
        &
        (df["rna_log2FC"] > 0),
        "rna_direction"
    ] = "Up"

    df.loc[
        (df["rna_sig"] == True)
        &
        (df["rna_log2FC"] < 0),
        "rna_direction"
    ] = "Down"


    # -----------------------------
    # Protein direction
    # -----------------------------
    df["protein_direction"] = "No change"

    df.loc[
        (df["protein_significance"] == "Significant")
        &
        (df["protein_log2FC"] > 0),
        "protein_direction"
    ] = "Up"

    df.loc[
        (df["protein_significance"] == "Significant")
        &
        (df["protein_log2FC"] < 0),
        "protein_direction"
    ] = "Down"


    return df

In [35]:
goblet_merged = assign_directions(goblet_merged)
club_merged = assign_directions(club_merged)
multiciliated_merged = assign_directions(multiciliated_merged)

## 6. RNA-protein concordance analysis

In [44]:
def add_concordance_flags(df):
    """
    Assign RNA and protein directionality and RNA-protein concordance categories
    for the GSE229202 integration.

    RNA direction:
    - significant + log2FC > 0 = Up
    - significant + log2FC < 0 = Down
    - not significant = No_change

    Protein direction:
    - significant + log2FC > 0 = Up
    - significant + log2FC < 0 = Down
    - not significant = No_change
    """

    df = df.copy()

    # -----------------------------
    # RNA direction
    # -----------------------------

    df["rna_direction"] = "No_change"

    df.loc[
        (df["rna_sig"] == True) &
        (df["rna_log2FC"] > 0),
        "rna_direction"
    ] = "Up"

    df.loc[
        (df["rna_sig"] == True) &
        (df["rna_log2FC"] < 0),
        "rna_direction"
    ] = "Down"


    # -----------------------------
    # Protein direction
    # -----------------------------

    df["protein_direction"] = "No_change"

    df.loc[
        (df["protein_significance"] == "Significant") &
        (df["protein_log2FC"] > 0),
        "protein_direction"
    ] = "Up"

    df.loc[
        (df["protein_significance"] == "Significant") &
        (df["protein_log2FC"] < 0),
        "protein_direction"
    ] = "Down"


    # -----------------------------
    # Concordance categories
    # -----------------------------

    df["concordance"] = np.select(
        [
            (df["rna_direction"] == "Up") &
            (df["protein_direction"] == "Up"),

            (df["rna_direction"] == "Down") &
            (df["protein_direction"] == "Down"),

            (df["rna_direction"] == "Up") &
            (df["protein_direction"] == "Down"),

            (df["rna_direction"] == "Down") &
            (df["protein_direction"] == "Up"),

            (df["rna_direction"] == "No_change") &
            (df["protein_direction"].isin(["Up", "Down"])),

            (df["protein_direction"] == "No_change") &
            (df["rna_direction"].isin(["Up", "Down"])),

            (df["rna_direction"] == "No_change") &
            (df["protein_direction"] == "No_change")
        ],
        [
            "Concordant Up",
            "Concordant Down",
            "Complete Discordant",
            "Complete Discordant",
            "Partial Discordant Protein-only",
            "Partial Discordant RNA-only",
            "Both No change"
        ],
        default="Other"
    )

    return df

In [45]:
goblet_final = add_concordance_flags(goblet_merged)
club_final = add_concordance_flags(club_merged)
multiciliated_final = add_concordance_flags(multiciliated_merged)

In [46]:
print("Goblet")
print(goblet_final["concordance"].value_counts())

print("\nClub")
print(club_final["concordance"].value_counts())

print("\nMulticiliated")
print(multiciliated_final["concordance"].value_counts())

Goblet
concordance
Partial Discordant Protein-only    3637
Both No change                     3409
Concordant Up                       948
Partial Discordant RNA-only         933
Concordant Down                     448
Complete Discordant                 317
Name: count, dtype: int64

Club
concordance
Partial Discordant RNA-only        3376
Concordant Down                    2189
Complete Discordant                1581
Both No change                     1421
Partial Discordant Protein-only     974
Concordant Up                       151
Name: count, dtype: int64

Multiciliated
concordance
Both No change                     6627
Partial Discordant Protein-only    2129
Partial Discordant RNA-only         554
Concordant Down                     150
Concordant Up                       140
Complete Discordant                  91
Name: count, dtype: int64


Check CLCA2 and calcium-signalling proteins

In [47]:
candidate_genes = [
    "CLCA2",
    "ORAI1",
    "STIM1",
    "ITPR2",
    "ITPR3"
]

In [48]:
goblet_candidates = goblet_final[
    goblet_final["gene"].isin(candidate_genes)
][
    [
        "gene",
        "rna_log2FC",
        "rna_padj",
        "rna_direction",
        "protein_log2FC",
        "protein_direction",
        "concordance"
    ]
].copy()

goblet_candidates["cell_type"] = "Goblet"

In [49]:
club_candidates = club_final[
    club_final["gene"].isin(candidate_genes)
][
    [
        "gene",
        "rna_log2FC",
        "rna_padj",
        "rna_direction",
        "protein_log2FC",
        "protein_direction",
        "concordance"
    ]
].copy()

club_candidates["cell_type"] = "Club"

In [50]:
multiciliated_candidates = multiciliated_final[
    multiciliated_final["gene"].isin(candidate_genes)
][
    [
        "gene",
        "rna_log2FC",
        "rna_padj",
        "rna_direction",
        "protein_log2FC",
        "protein_direction",
        "concordance"
    ]
].copy()

multiciliated_candidates["cell_type"] = "Multiciliated"

In [51]:
candidate_summary = pd.concat(
    [
        goblet_candidates,
        club_candidates,
        multiciliated_candidates
    ],
    ignore_index=True
)

candidate_summary

,gene,rna_log2FC,rna_padj,rna_direction,protein_log2FC,protein_direction,concordance,cell_type
0,STIM1,0.244112,2.453750e-01,No_change,1.123525,Up,Partial Discordant Protein-only,Goblet
1,ITPR3,0.649092,1.588888e-05,Up,2.829960,Up,Concordant Up,Goblet
2,ITPR2,-1.223843,2.214463e-01,No_change,2.158568,Up,Partial Discordant Protein-only,Goblet
3,CLCA2,2.920387,8.607152e-01,No_change,3.549875,Up,Partial Discordant Protein-only,Goblet
4,ORAI1,1.241566,1.929202e-19,Up,2.516454,Up,Concordant Up,Goblet
5,ITPR3,-0.477416,7.596892e-04,Down,1.710436,Up,Complete Discordant,Club
6,STIM1,-0.590388,1.146798e-05,Down,0.899264,Up,Complete Discordant,Club
7,ITPR2,-0.624068,3.741957e-01,No_change,1.602743,Up,Partial Discordant Protein-only,Club
8,CLCA2,2.433199,1.000000e+00,No_change,3.711392,Up,Partial Discordant Protein-only,Club
9,ORAI1,-0.315333,3.281641e-03,Down,2.012068,Up,Complete Discordant,Club


In [52]:
goblet_final.to_csv(
    "results/GSE229202_goblet_integration.csv",
    index=False
)

club_final.to_csv(
    "results/GSE229202_club_integration.csv",
    index=False
)

multiciliated_final.to_csv(
    "results/GSE229202_multiciliated_integration.csv",
    index=False
)